# Ship Engine Anomaly Detection - Example Notebook

This notebook demonstrates how to use the ship engine anomaly detection system interactively.

## 1. Setup and Imports

In [ ]:
import sys
from pathlib import Path

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from anomaly_detection import AnomalyDetector, create_anomaly_summary
from preprocessing import load_data, preprocess_data, get_feature_statistics
from utils import load_config, setup_logging, download_file, ensure_directories
from visualization import create_all_visualizations

%matplotlib inline
sns.set_style('whitegrid')

## 2. Load Configuration

In [ ]:
# Load configuration
config_path = Path.cwd().parent / 'config.yaml'
config = load_config(str(config_path))

# Setup logging
logger = setup_logging(config)

# Ensure directories exist
ensure_directories(config)

print("Configuration loaded successfully!")

## 3. Load and Explore Data

In [ ]:
# Download data if needed
data_path = Path.cwd().parent / config['data']['raw_path']

if not data_path.exists():
    print("Downloading dataset...")
    download_file(config['data']['url'], str(data_path), config, logger)
    print("Download complete!")
else:
    print(f"Using existing dataset: {data_path}")

# Load data
df = load_data(str(data_path), config, logger)
print(f"\nLoaded {len(df)} samples with {len(df.columns)} features")

In [ ]:
# Display first few rows
df.head()

In [ ]:
# Display feature statistics
stats = get_feature_statistics(df, logger)
stats

## 4. Preprocess Data

In [ ]:
# Preprocess data
df_processed, scaler = preprocess_data(df, config, logger)
print(f"\nPreprocessed data shape: {df_processed.shape}")
df_processed.head()

## 5. Anomaly Detection

In [ ]:
# Initialize detector
detector = AnomalyDetector(config, logger)

# Run all detection methods
results, pca_data = detector.detect_all_anomalies(df_processed)

In [ ]:
# Display summary
summary = create_anomaly_summary(results)
summary

## 6. Visualizations

### IQR Method Results

In [ ]:
if 'iqr' in results:
    iqr_results = results['iqr']
    print(f"IQR Method: {iqr_results['n_anomalies']} anomalies detected ({iqr_results['percentage']:.2f}%)")
    print(f"\nMost anomalous features: {', '.join(iqr_results['top_features'])}")

### PCA Scatter Plots

In [ ]:
if pca_data is not None:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    methods = [
        ('IQR', 'iqr'),
        ('One-Class SVM', 'one_class_svm'),
        ('Isolation Forest', 'isolation_forest')
    ]
    
    for idx, (name, key) in enumerate(methods):
        if key in results:
            labels = results[key]['labels']
            normal_mask = labels == 1
            anomaly_mask = labels == -1
            
            axes[idx].scatter(pca_data[normal_mask, 0], pca_data[normal_mask, 1], 
                            c='blue', alpha=0.5, s=20, label='Normal')
            axes[idx].scatter(pca_data[anomaly_mask, 0], pca_data[anomaly_mask, 1], 
                            c='red', alpha=0.7, s=50, marker='x', label='Anomaly')
            axes[idx].set_xlabel('First Principal Component')
            axes[idx].set_ylabel('Second Principal Component')
            axes[idx].set_title(f'Anomaly Detection: {name}')
            axes[idx].legend()
            axes[idx].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

### Feature Distributions

In [ ]:
# Plot feature distributions with anomalies highlighted
if 'iqr' in results:
    labels = results['iqr']['labels']
    
    numeric_cols = df_processed.select_dtypes(include=[np.number]).columns
    n_cols = len(numeric_cols)
    n_plot_cols = 3
    n_plot_rows = (n_cols + n_plot_cols - 1) // n_plot_cols
    
    fig, axes = plt.subplots(n_plot_rows, n_plot_cols, figsize=(15, 5 * n_plot_rows))
    axes = axes.flatten() if n_cols > 1 else [axes]
    
    for idx, col in enumerate(numeric_cols):
        ax = axes[idx]
        
        # Plot normal points
        ax.hist(df_processed[col][labels == 1], bins=30, alpha=0.7, 
               color='blue', label='Normal', edgecolor='black')
        
        # Plot anomalies
        anomaly_values = df_processed[col][labels == -1]
        if len(anomaly_values) > 0:
            ax.hist(anomaly_values, bins=30, alpha=0.7, 
                   color='red', label='Anomaly', edgecolor='black')
        
        ax.set_xlabel(col)
        ax.set_ylabel('Frequency')
        ax.set_title(f'Distribution: {col}')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    # Hide unused subplots
    for idx in range(n_cols, len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.show()

## 7. Recommendations

In [ ]:
if 'recommendations' in results:
    print("\nRECOMMENDATIONS:")
    print("=" * 80)
    for i, rec in enumerate(results['recommendations'], 1):
        print(f"{i}. {rec}")

## 8. Export Results

In [ ]:
# Create DataFrame with anomaly labels
df_with_labels = df_processed.copy()

if 'iqr' in results:
    df_with_labels['anomaly_iqr'] = results['iqr']['labels']
if 'one_class_svm' in results:
    df_with_labels['anomaly_svm'] = results['one_class_svm']['labels']
if 'isolation_forest' in results:
    df_with_labels['anomaly_if'] = results['isolation_forest']['labels']

# Display samples flagged as anomalies by all methods
all_anomalies = (
    (df_with_labels['anomaly_iqr'] == -1) & 
    (df_with_labels['anomaly_svm'] == -1) & 
    (df_with_labels['anomaly_if'] == -1)
)

print(f"Samples flagged by all methods: {all_anomalies.sum()}")
if all_anomalies.sum() > 0:
    df_with_labels[all_anomalies].head(10)

In [ ]:
# Save results
output_path = Path.cwd().parent / 'results' / 'anomaly_results.csv'
df_with_labels.to_csv(output_path, index=False)
print(f"Results saved to {output_path}")